# INSTALL DEPENDENCIES

In [1]:
INITAL_SETUP = False

if INITAL_SETUP:
    %pip install fastkaggle
    %pip install kaggle
    %pip install dotenv
    %pip install ipdb


In [ ]:
import fastkaggle
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
import os
from fastai.tabular import *
from fastai.tabular.all import *

PATH_LOCAL_PATH_DATA_STORAGE = Path(r'D:\kaggle_data')
COMPETITION_NAME = 'titanic'


# CHECK IF RUNNING ON KAGGLE OR ELSEWHERE

In [3]:
'Kaggle' if fastkaggle.iskaggle else 'Not Kaggle'

'Not Kaggle'

# PREPARE KAGGLE DATA

In [4]:
if fastkaggle.iskaggle:
    path = os.path.join(Path('../input'), COMPETITION_NAME) # generates path containing competition name
    
else:
    path_temp = Path(COMPETITION_NAME)
    
    # check if data is missing from the desired location
    path_local_competition_data = os.path.join(Path(PATH_LOCAL_PATH_DATA_STORAGE), COMPETITION_NAME) # generates path containing competition name
    if not os.path.isdir(path_local_competition_data):
        fastkaggle.setup_comp('titanic') # download data to temp location
        dest = shutil.move(path_temp, PATH_LOCAL_PATH_DATA_STORAGE) # move the data to the correct location
    
    path =path_local_competition_data # update path to point to folder containing data

print("Data located at: {}".format(str(path)))
    
        

Data located at: D:\kaggle_data\titanic


In [5]:
path_train = os.path.join(path, 'train.csv')

train_df = pd.read_csv(path_train)
train_df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


# Prepare dataset

We will apply the following transformations on the dataset.

* Tokenize the names. For example, "Braund, Mr. Owen Harris" will become ["Braund", "Mr.", "Owen", "Harris"].
* Extract any prefix in the ticket. For example ticket "STON/O2. 3101282" will become "STON/O2." and 3101282.


In [6]:
train_df.isnull().any()

PassengerId    False
Survived       False
Pclass         False
Name           False
Sex            False
Age             True
SibSp          False
Parch          False
Ticket         False
Fare           False
Cabin           True
Embarked        True
dtype: bool

There are n/a values located in
* age
* cabin 
* embarked

We need to clean these up.

In [7]:
def preprocess(df):
    df = df.copy()

    df.replace({'Embarked':np.nan}, inplace=True)
    df.replace({'Age':np.nan}, inplace=True)
    df.replace({'Cabin':np.nan}, inplace=True)
    
    # fill nan with assumed values
    df.fillna({'Embarked': 'S'}, inplace=True)
    df.fillna({'Fare': np.mean(df['Fare'])}, inplace=True)
    age_avg = df['Age'].mean()
    age_std = df['Age'].std()
    df.fillna(
        {'Age': np.random.randint(age_avg - age_std, age_avg + age_std)}, inplace=True)
       
    # Extract the deck letter from the cabin
    df['Deck'] = df['Cabin'].str[0]
    df.replace({'Unknown':''}, inplace=True)
    
    # hot key for female
    df.replace(
        {'Sex':(['male','female'], [0, 1])}, 
        inplace=True)
    
    df['Embarked'] = df['Embarked'].map( {'S': 0, 'C': 1, 'Q': 2} ).astype('Int64')
    df['family'] = (df['SibSp'] + df['Parch'])
    df['isAlone'] = 1
    df.loc[df['family'] > 0, 'isAlone'] = 0
    delete_columns = ['Name','Ticket','Cabin']
    df.drop(delete_columns, axis=1, inplace=True)
                     
    return df
    
preprocessed_train_df = preprocess(train_df)
preprocessed_train_df.head(5)

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Deck,family,isAlone
0,1,0,3,male,22.0,1,0,7.2500,0,NaN,1,0
1,2,1,1,female,38.0,1,0,71.2833,1,C,1,0
2,3,1,3,female,26.0,0,0,7.9250,0,NaN,0,1
3,4,1,1,female,35.0,1,0,53.1000,0,C,1,0
4,5,0,3,male,35.0,0,0,8.0500,0,NaN,0,1


# Prepare dataloaders

In [ ]:
dep_var = "Survived"
cat_names= ['Pclass', 'Sex', 'Embarked', 'Deck'] # 'isAlone'
cont_names = ['Fare', 'Age', 'SibSp', 'Parch'] # family

dls1 = all.TabularDataLoaders.from_df(preprocessed_train_df, 
                    cat_names= cat_names,
                    cont_names = cont_names,
                    y_names= dep_var,
                    procs = [Categorify]
                                 )


AttributeError: 'builtin_function_or_method' object has no attribute 'TabularDataLoaders'